# Week 1: Traditional Computer Vision for Cancer Detection
**FURP 2026 — Cancer Detection Using Artificial Intelligence**  
**Instructor: Prof. Elio Espejo**

---

## 本周目标
1. 生成合成乳腺X光图像（高斯分布 + 三角函数纹理）
2. 提取19个数学特征（统计 / 边缘 / 纹理 / 形态 / 频率）
3. 训练随机森林分类器（信息熵 + Bootstrap + 投票）
4. AI Assistant Tasks 1-4（可视化 / GLCM / 性能分析 / 质量评估）

## 环境准备

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'seaborn', 'scikit-image', 'opencv-python',
                'scikit-learn', 'matplotlib', '--quiet'], capture_output=True)
print('✅ 依赖库检查完成')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
np.random.seed(42)
print('✅ 所有库导入成功！')

---
## Step 1: 合成乳腺X光图像生成

**数学原理：**
$$I(x,y) = \mathcal{N}(\mu=120, \sigma=25) + 20\sin\left(\frac{x}{30}\right)\cos\left(\frac{y}{25}\right) + \mathcal{N}(0,10)$$

癌症肿块：$dx^2 + dy^2 < r^2$ 区域内像素 $+60 + \mathcal{N}(0,5)$

In [ ]:
def create_synthetic_mammograms(num_samples=500):
    print(f'正在创建 {num_samples} 张合成医学图像...')
    np.random.seed(42)
    images, labels = [], []
    image_size = 256
    y, x = np.ogrid[:image_size, :image_size]

    for i in range(num_samples):
        # 正态分布模拟乳腺组织 N(120, 25)
        image = np.random.normal(120, 25, (image_size, image_size))
        # 三角函数模拟纤维纹理
        tissue_pattern = 20 * np.sin(x/30) * np.cos(y/25)
        image += tissue_pattern

        has_cancer = np.random.random() < 0.3

        if has_cancer:
            center_x = np.random.randint(40, image_size-40)
            center_y = np.random.randint(40, image_size-40)
            radius = np.random.randint(15, 25)
            for dy in range(-radius, radius):
                for dx in range(-radius, radius):
                    if dx**2 + dy**2 < radius**2:
                        irregularity = np.random.normal(0, 5)
                        if (0 <= center_y+dy < image_size and
                            0 <= center_x+dx < image_size):
                            image[center_y+dy, center_x+dx] += 60 + irregularity
            labels.append(1)
        else:
            if np.random.random() < 0.2:
                center_x = np.random.randint(30, image_size-30)
                center_y = np.random.randint(30, image_size-30)
                benign_radius = np.random.randint(5, 12)
                benign_mask = ((x-center_x)**2 + (y-center_y)**2) < benign_radius**2
                image[benign_mask] += 25
            labels.append(0)

        noise = np.random.normal(0, 10, image.shape)
        image += noise
        image = np.clip(image, 0, 255).astype(np.uint8)
        images.append(image)

        if (i + 1) % 100 == 0:
            print(f'  已生成 {i+1}/{num_samples} 张图像')

    images = np.array(images)
    labels = np.array(labels)
    cancer_count = np.sum(labels)
    print(f'\n📊 数据集统计：')
    print(f'  总图像数：{len(images)}')
    print(f'  癌症图像：{cancer_count} ({cancer_count/len(labels)*100:.1f}%)')
    print(f'  正常图像：{len(labels)-cancer_count}')
    return images, labels

images, labels = create_synthetic_mammograms(500)

In [ ]:
# 可视化样本图像
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('合成乳腺X光图像样本', fontsize=16, fontweight='bold')

normal_idx = np.where(labels == 0)[0][:5]
cancer_idx = np.where(labels == 1)[0][:5]

for i, idx in enumerate(normal_idx):
    axes[0, i].imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(f'Normal #{idx}', color='#1D9E75', fontsize=10)
    axes[0, i].axis('off')

for i, idx in enumerate(cancer_idx):
    axes[1, i].imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Cancer #{idx}', color='#E24B4A', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Normal', fontsize=12, color='#1D9E75', fontweight='bold')
axes[1, 0].set_ylabel('Cancer', fontsize=12, color='#E24B4A', fontweight='bold')
plt.tight_layout()
plt.savefig('viz_1_sample_images.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 2: 数学特征提取

| 类别 | 特征 | 数量 | 数学工具 |
|------|------|------|----------|
| 统计 | 均值、标准差、偏度、峰度... | 7 | 概率论 |
| 边缘 | 边缘密度、强度、方差 | 3 | Sobel 算子（离散梯度）|
| 纹理 | 均匀度、熵 | 2 | LBP 局部二值模式 |
| 形态 | 圆度、长宽比、凸度 | 3 | 轮廓分析 |
| 频率 | FFT均值、标准差、低频、高频能量 | 4 | 二维傅里叶变换 |

**Sobel边缘公式：** $G = \sqrt{G_x^2 + G_y^2}$  
**圆度公式：** $C = \frac{4\pi A}{P^2}$（完美圆=1.0）  
**纹理熵：** $H = -\sum_k h_k \log_2 h_k$

In [ ]:
def extract_comprehensive_features(image):
    features = {}

    # === 1. 统计特征 ===
    features['mean_intensity']   = np.mean(image)
    features['std_intensity']    = np.std(image)
    features['intensity_range']  = np.max(image) - np.min(image)
    normalized_image = (image - features['mean_intensity']) / (features['std_intensity'] + 1e-7)
    features['skewness']          = np.mean(normalized_image**3)
    features['kurtosis']          = np.mean(normalized_image**4)
    features['median_intensity']  = np.median(image)
    features['iqr']               = np.percentile(image, 75) - np.percentile(image, 25)

    # === 2. 边缘特征（Sobel算子 / 离散梯度）===
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    edge_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    features['edge_density']  = np.mean(edge_magnitude > 50)
    features['edge_strength'] = np.mean(edge_magnitude)
    features['edge_variance'] = np.var(edge_magnitude)

    # === 3. 纹理特征（LBP）===
    lbp = local_binary_pattern(image, 8, 1, method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0, 9))
    lbp_hist = lbp_hist.astype(float) / (lbp_hist.sum() + 1e-7)
    features['texture_uniformity'] = np.sum(lbp_hist**2)
    features['texture_entropy']    = -np.sum(lbp_hist * np.log2(lbp_hist + 1e-7))

    # === 4. 形态特征 ===
    _, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest  = max(contours, key=cv2.contourArea)
        area     = cv2.contourArea(largest)
        perimeter = cv2.arcLength(largest, True)
        features['circularity'] = 4*np.pi*area/(perimeter**2) if perimeter > 0 else 0
        xc, yc, w, h = cv2.boundingRect(largest)
        features['aspect_ratio'] = w/h if h > 0 else 0
        hull = cv2.convexHull(largest)
        hull_area = cv2.contourArea(hull)
        features['solidity'] = area/hull_area if hull_area > 0 else 0
    else:
        features.update({'circularity': 0, 'aspect_ratio': 0, 'solidity': 0})

    # === 5. 频率特征（2D FFT）===
    fft       = np.fft.fft2(image)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.abs(fft_shift)
    features['fft_mean'] = np.mean(magnitude)
    features['fft_std']  = np.std(magnitude)
    cy, cx = magnitude.shape[0]//2, magnitude.shape[1]//2
    y_c, x_c = np.ogrid[:magnitude.shape[0], :magnitude.shape[1]]
    low_r = min(cx, cy) // 4
    features['low_freq_energy']  = np.sum(magnitude[(x_c-cx)**2 + (y_c-cy)**2 <= low_r**2])
    high_r = min(cx, cy) // 2
    features['high_freq_energy'] = np.sum(magnitude[(x_c-cx)**2 + (y_c-cy)**2 >= high_r**2])

    return list(features.values())


print('开始提取特征...')
features_list = []
for i, img in enumerate(images):
    features_list.append(extract_comprehensive_features(img))
    if (i + 1) % 100 == 0:
        print(f'  已处理 {i+1}/500 张图像')

features_array = np.array(features_list)
print(f'\n✅ 特征提取完成！')
print(f'特征矩阵形状：{features_array.shape}')
print(f'→ 500张图像，每张 {features_array.shape[1]} 个特征')

In [ ]:
# 特征分布对比可视化
feature_names = [
    'Mean Intensity', 'Std Intensity', 'Intensity Range',
    'Skewness', 'Kurtosis', 'Median Intensity', 'IQR',
    'Edge Density', 'Edge Strength', 'Edge Variance',
    'Texture Uniformity', 'Texture Entropy',
    'Circularity', 'Aspect Ratio', 'Solidity',
    'FFT Mean', 'FFT Std', 'Low Freq Energy', 'High Freq Energy'
]

normal_features = features_array[labels == 0]
cancer_features = features_array[labels == 1]
top_features = [0, 1, 7, 8, 11, 12, 14, 17, 18]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Feature Distribution: Normal vs Cancer', fontsize=16, fontweight='bold')

for plot_idx, feat_idx in enumerate(top_features):
    ax = axes[plot_idx // 3][plot_idx % 3]
    ax.hist(normal_features[:, feat_idx], bins=30, alpha=0.6,
            color='#1D9E75', label='Normal', density=True)
    ax.hist(cancer_features[:, feat_idx], bins=30, alpha=0.6,
            color='#E24B4A', label='Cancer', density=True)
    ax.axvline(np.mean(normal_features[:, feat_idx]),
               color='#1D9E75', linestyle='--', linewidth=2)
    ax.axvline(np.mean(cancer_features[:, feat_idx]),
               color='#E24B4A', linestyle='--', linewidth=2)
    ax.set_title(feature_names[feat_idx], fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('viz_2_feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 3: 随机森林分类器

**核心公式：**

信息熵：$H(S) = -\sum_c p_c \log_2 p_c$

信息增益：$\text{IG} = H(S) - \frac{|S_L|}{|S|}H(S_L) - \frac{|S_R|}{|S|}H(S_R)$

标准化：$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$

最终预测：$\hat{y} = \text{argmax}_c \sum_{t=1}^{100} \mathbf{1}[h_t(x)=c]$

In [ ]:
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(
    features_array, labels,
    test_size=0.2, random_state=42, stratify=labels
)
print(f'训练集：{len(X_train)} 张 | 测试集：{len(X_test)} 张')

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('特征已标准化（零均值，单位方差）')

# 训练随机森林
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced'
)
print('\n正在训练随机森林（100棵决策树）...')
clf.fit(X_train_scaled, y_train)

# 预测
y_pred  = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

# 计算指标
accuracy    = accuracy_score(y_test, y_pred)
precision   = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1          = f1_score(y_test, y_pred)
roc_auc     = roc_auc_score(y_test, y_proba)
tn = np.sum((y_pred==0)&(y_test==0))
fp = np.sum((y_pred==1)&(y_test==0))
specificity = tn/(tn+fp) if (tn+fp)>0 else 0

print(f'\n{"="*50}')
print(f'         Week 1 实验结果')
print(f'{"="*50}')
print(f'准确率      : {accuracy*100:.1f}%')
print(f'精确率      : {precision*100:.1f}%')
print(f'敏感性      : {sensitivity*100:.1f}%  ← 医学最重要！')
print(f'特异性      : {specificity*100:.1f}%')
print(f'F1 分数     : {f1:.3f}')
print(f'AUC-ROC    : {roc_auc:.3f}')
print(f'{"="*50}')

In [ ]:
# 5折交叉验证（检测过拟合）
print('正在运行 5 折交叉验证...')
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    features_array, labels, cv=5, scoring='roc_auc'
)
print(f'各折 AUC: {[f"{s:.3f}" for s in cv_scores]}')
print(f'平均 AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

---
## AI Assistant Task 1: Advanced Visualization
ROC曲线 / 混淆矩阵 / Precision-Recall曲线 / 特征重要性 / 预测置信度

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 12))
fig.suptitle('Week 1: Comprehensive Medical AI Analysis',
             fontsize=18, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── 1. 混淆矩阵 ──
ax1 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Cancer'],
            yticklabels=['Normal','Cancer'],
            ax=ax1, cbar=False,
            annot_kws={'size':16,'weight':'bold'})
tn_v, fp_v, fn_v, tp_v = cm.ravel()
ax1.set_title('Confusion Matrix', fontweight='bold', fontsize=13)
ax1.set_xlabel('Predicted', fontsize=11)
ax1.set_ylabel('Actual', fontsize=11)
ax1.text(2.2, 1.0,
         f'TP={tp_v}  FP={fp_v}\nFN={fn_v}  TN={tn_v}',
         fontsize=10, va='center',
         bbox=dict(boxstyle='round', facecolor='#E6F1FB', alpha=0.8))

# ── 2. ROC 曲线 ──
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='#185FA5', lw=2.5,
         label=f'AUC = {roc_auc_val:.3f}')
ax2.fill_between(fpr, tpr, alpha=0.08, color='#185FA5')
ax2.plot([0,1],[0,1],'k--',lw=1.5,alpha=0.5,label='Random')
opt_idx = np.argmax(tpr - fpr)
ax2.scatter(fpr[opt_idx], tpr[opt_idx], color='#E24B4A', s=120, zorder=5,
            label=f'Optimal @ {thresholds[opt_idx]:.2f}')
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curve', fontweight='bold', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# ── 3. Precision-Recall 曲线 ──
ax3 = fig.add_subplot(gs[0, 2])
prec, rec, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(rec, prec)
ax3.plot(rec, prec, color='#1D9E75', lw=2.5, label=f'PR AUC = {pr_auc:.3f}')
ax3.fill_between(rec, prec, alpha=0.08, color='#1D9E75')
baseline = np.sum(y_test)/len(y_test)
ax3.axhline(y=baseline, color='#E24B4A', linestyle='--',
            lw=1.5, label=f'Baseline={baseline:.2f}')
ax3.set_xlabel('Recall (Sensitivity)', fontsize=11)
ax3.set_ylabel('Precision', fontsize=11)
ax3.set_title('Precision-Recall Curve', fontweight='bold', fontsize=13)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# ── 4. 特征重要性 ──
ax4 = fig.add_subplot(gs[1, 0:2])
importances = clf.feature_importances_
std_imp = np.std([t.feature_importances_ for t in clf.estimators_], axis=0)
idx = np.argsort(importances)[::-1]
short_names = ['Mean','Std','Range','Skew','Kurt','Med','IQR',
               'EdgeD','EdgeS','EdgeV','TexU','TexE',
               'Circ','AR','Solid','FFTM','FFTS','LowF','HighF']
colors = ['#185FA5' if importances[i]>0.08
          else '#5DCAA5' if importances[i]>0.04
          else '#B4B2A9' for i in idx]
ax4.bar(range(19), importances[idx], color=colors,
        yerr=std_imp[idx], capsize=3, alpha=0.85)
ax4.set_xticks(range(19))
ax4.set_xticklabels([short_names[i] for i in idx],
                     rotation=45, ha='right', fontsize=9)
ax4.set_ylabel('Importance Score', fontsize=11)
ax4.set_title('Feature Importance with Std Dev', fontweight='bold', fontsize=13)
ax4.grid(True, alpha=0.3, axis='y')

# ── 5. 预测置信度 ──
ax5 = fig.add_subplot(gs[1, 2])
sorted_idx = np.argsort(y_proba)
ax5.scatter(range(len(y_proba)), y_proba[sorted_idx],
            c=['#E24B4A' if y==1 else '#1D9E75' for y in y_test[sorted_idx]],
            alpha=0.7, s=30)
ax5.axhline(y=0.5, color='black', linestyle='--', lw=1.5,
            label='Threshold=0.5')
ax5.fill_between(range(len(y_proba)), 0.3, 0.7,
                 alpha=0.1, color='orange', label='Uncertain zone')
ax5.set_xlabel('Sample (sorted by probability)', fontsize=11)
ax5.set_ylabel('Cancer Probability', fontsize=11)
ax5.set_title('Prediction Confidence', fontweight='bold', fontsize=13)
from matplotlib.patches import Patch
ax5.legend(handles=[
    Patch(facecolor='#E24B4A', label='Actual Cancer'),
    Patch(facecolor='#1D9E75', label='Actual Normal')
], fontsize=9, loc='upper left')
ax5.grid(True, alpha=0.3)

plt.savefig('task1_comprehensive_viz.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 1 完成！已保存：task1_comprehensive_viz.png')

---
## AI Assistant Task 2: GLCM 纹理特征

**数学定义：**
$$P(i,j,d,\theta) = \text{在方向}\theta\text{、距离}d\text{下，像素值}i\text{旁边出现像素值}j\text{的概率}$$

| 特征 | 公式 |
|------|------|
| Contrast | $\sum(i-j)^2 \cdot P(i,j)$ |
| Correlation | $\sum\frac{(i-\mu_i)(j-\mu_j)P(i,j)}{\sigma_i\sigma_j}$ |
| Energy | $\sum P(i,j)^2$ |
| Homogeneity | $\sum\frac{P(i,j)}{1+|i-j|}$ |

In [ ]:
def compute_glcm_features(image):
    image_32 = (image / 8).astype(np.uint8)
    glcm = graycomatrix(image_32, distances=[1,2],
                        angles=[0, np.pi/4, np.pi/2],
                        levels=32, symmetric=True, normed=True)
    features = {}
    for prop in ['contrast','correlation','energy','homogeneity']:
        values = graycoprops(glcm, prop)
        features[f'{prop}_mean'] = np.mean(values)
        features[f'{prop}_std']  = np.std(values)
    return features

n = 50
normal_idx = np.where(labels == 0)[0][:n]
cancer_idx = np.where(labels == 1)[0][:n]

print('计算GLCM特征...')
normal_glcm = np.array([[v for v in compute_glcm_features(images[i]).values()]
                         for i in normal_idx])
cancer_glcm = np.array([[v for v in compute_glcm_features(images[i]).values()]
                         for i in cancer_idx])
feat_names = list(compute_glcm_features(images[0]).keys())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('GLCM Features: Normal vs Cancer', fontsize=16, fontweight='bold')

for i, name in enumerate(feat_names[:8]):
    ax = axes[i//4][i%4]
    ax.hist(normal_glcm[:, i], bins=20, alpha=0.6,
            color='#1D9E75', label='Normal', density=True)
    ax.hist(cancer_glcm[:, i], bins=20, alpha=0.6,
            color='#E24B4A', label='Cancer', density=True)
    ax.axvline(np.mean(normal_glcm[:, i]), color='#1D9E75',
               linestyle='--', linewidth=2)
    ax.axvline(np.mean(cancer_glcm[:, i]), color='#E24B4A',
               linestyle='--', linewidth=2)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task2_glcm_features.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nGLCM特征对比（均值）：')
print(f'{"特征":<25} {"正常":>10} {"癌症":>10} {"差异%":>10}')
print('-' * 57)
for i, name in enumerate(feat_names):
    n_m = np.mean(normal_glcm[:, i])
    c_m = np.mean(cancer_glcm[:, i])
    diff = abs(c_m-n_m)/(abs(n_m)+1e-7)*100
    print(f'{name:<25} {n_m:>10.4f} {c_m:>10.4f} {diff:>9.1f}%')

print('\n✅ Task 2 完成！已保存：task2_glcm_features.png')

---
## AI Assistant Task 3: 统计性能分析
Bootstrap置信区间 / 5折交叉验证 / 敏感性-特异性权衡 / 临床解读

In [ ]:
def bootstrap_ci(y_true, y_pred, y_proba, metric_fn, n=1000, ci=0.95):
    """Bootstrap置信区间：有放回重采样n次，取2.5%~97.5%分位数"""
    scores = []
    size = len(y_true)
    for _ in range(n):
        idx = np.random.choice(size, size, replace=True)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx], y_proba[idx]))
        except:
            continue
    scores = np.array(scores)
    alpha = (1-ci)/2
    return np.mean(scores), np.percentile(scores, alpha*100), np.percentile(scores, (1-alpha)*100)

print('='*60)
print('    Week 1 完整统计性能分析')
print('='*60)

# 5折交叉验证
print('\n[1] 5折分层交叉验证')
print('-'*40)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_s = scaler.fit_transform(features_array)

cv_results = {}
for name, scoring in [('AUC-ROC','roc_auc'),('Accuracy','accuracy'),
                       ('Sensitivity','recall'),('Precision','precision')]:
    s = cross_val_score(clf, X_s, labels, cv=cv, scoring=scoring)
    cv_results[name] = s
    print(f'{name:<15}: {s.mean():.3f} ± {s.std():.3f}  '
          f'95%CI [{np.percentile(s,2.5):.3f}, {np.percentile(s,97.5):.3f}]')

# Bootstrap置信区间
print('\n[2] Bootstrap 置信区间 (n=1000)')
print('-'*40)
sens_fn = lambda yt,yp,_: recall_score(yt,yp)
spec_fn = lambda yt,yp,_: (np.sum((yp==0)&(yt==0)) /
                            max(np.sum((yp==0)&(yt==0))+np.sum((yp==1)&(yt==0)),1))
auc_fn  = lambda yt,_,ypr: roc_auc_score(yt,ypr)

for name, fn in [('Sensitivity',sens_fn),('Specificity',spec_fn),('AUC-ROC',auc_fn)]:
    mean, low, high = bootstrap_ci(y_test, y_pred, y_proba, fn)
    print(f'{name:<15}: {mean:.3f}  95%CI [{low:.3f}, {high:.3f}]')

# 临床解读
print(f'\n[3] 临床意义解读')
print('-'*40)
print(f'敏感性 {sensitivity:.1%}：每100个癌症患者约{sensitivity*100:.0f}个被正确检测')
print(f'特异性 {specificity:.1%}：每100个正常患者约{specificity*100:.0f}个被正确判断')
print(f'\n医学权衡：漏诊(FN)代价 >> 误报(FP)代价 → 优先提高敏感性')
print(f'建议临床阈值：降至 0.3（提高敏感性至约95%）')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Performance Analysis', fontsize=16, fontweight='bold')

axes[0].boxplot(list(cv_results.values()),
                labels=list(cv_results.keys()),
                patch_artist=True,
                boxprops=dict(facecolor='#B5D4F4', color='#185FA5'),
                medianprops=dict(color='#E24B4A', linewidth=2))
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('5-Fold Cross Validation', fontweight='bold', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0.5, 1.05])

thresholds_range = np.linspace(0.1, 0.9, 50)
sensitivities, specificities = [], []
for t in thresholds_range:
    yp_t = (y_proba >= t).astype(int)
    sensitivities.append(recall_score(y_test, yp_t, zero_division=0))
    tn_t = np.sum((yp_t==0)&(y_test==0))
    fp_t = np.sum((yp_t==1)&(y_test==0))
    specificities.append(tn_t/(tn_t+fp_t) if (tn_t+fp_t)>0 else 0)

axes[1].plot(thresholds_range, sensitivities, 'o-',
             color='#E24B4A', lw=2, label='Sensitivity', markersize=4)
axes[1].plot(thresholds_range, specificities, 's-',
             color='#185FA5', lw=2, label='Specificity', markersize=4)
axes[1].axvline(x=0.5, color='gray', linestyle='--', lw=1.5, label='Default=0.5')
axes[1].axvline(x=0.3, color='orange', linestyle='--', lw=1.5, label='Clinical=0.3')
axes[1].set_xlabel('Decision Threshold', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Sensitivity-Specificity Trade-off', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_performance_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n✅ Task 3 完成！已保存：task3_performance_analysis.png')

---
## AI Assistant Task 4: 图像质量评估
质量检测（对比度/亮度/清晰度/信息量）/ CLAHE预处理 / 批量质量报告

In [ ]:
def assess_image_quality(image):
    issues, scores = [], {}

    # 对比度
    contrast = np.std(image)
    scores['contrast'] = min(1.0, contrast/50.0)
    if contrast < 15: issues.append(f'对比度过低({contrast:.1f})')

    # 亮度
    mean_b = np.mean(image)
    scores['brightness'] = max(0, 1.0 - abs(mean_b-128)/128)
    if mean_b < 30:  issues.append(f'图像过暗(均值={mean_b:.1f})')
    if mean_b > 220: issues.append(f'图像过亮(均值={mean_b:.1f})')

    # 清晰度（Laplacian方差）
    lap_var = cv2.Laplacian(image, cv2.CV_64F).var()
    scores['sharpness'] = min(1.0, lap_var/500.0)
    if lap_var < 10: issues.append(f'图像模糊(方差={lap_var:.1f})')

    # 有效性
    scores['validity'] = 0.0 if (np.isnan(image).any() or
                                  np.isinf(image).any()) else 1.0

    # 信息量（边缘密度）
    edges = cv2.Canny(image, 30, 100)
    edge_d = np.mean(edges > 0)
    scores['information'] = min(1.0, edge_d/0.1)
    if edge_d < 0.005: issues.append(f'信息量不足(边缘={edge_d:.4f})')

    overall = np.mean(list(scores.values()))
    rating = ('Excellent' if overall>0.8 else
              'Good'      if overall>0.6 else
              'Acceptable'if overall>0.4 else 'Poor')
    return {'overall_score': overall, 'quality_rating': rating,
            'issues': issues, 'is_acceptable': overall>0.4}


def preprocess_clahe(image):
    """CLAHE对比度增强 + 高斯降噪 + 归一化"""
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(image)
    denoised = cv2.GaussianBlur(enhanced, (3,3), 0)
    return cv2.normalize(denoised, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


print('开始批量图像质量检测...')
results = []
for i, img in enumerate(images):
    r = assess_image_quality(img)
    r['index'] = i
    results.append(r)
    if (i+1) % 100 == 0:
        print(f'  已检测 {i+1}/500 张')

scores_all = [r['overall_score'] for r in results]
ratings    = [r['quality_rating'] for r in results]
poor_count = sum(1 for r in results if not r['is_acceptable'])

print(f'\n{"="*50}')
print('数据集质量报告')
print(f'{"="*50}')
print(f'总图像数:   {len(images)}')
print(f'平均质量分: {np.mean(scores_all):.3f} ± {np.std(scores_all):.3f}')
print(f'低质量图像: {poor_count} 张 ({poor_count/len(images)*100:.1f}%)')
print('\n质量分布:')
for rating in ['Excellent','Good','Acceptable','Poor']:
    count = ratings.count(rating)
    bar = '█' * int(count/len(images)*40)
    print(f'  {rating:<12}: {count:>4} ({count/len(images)*100:>5.1f}%) {bar}')

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Dataset Quality Assessment', fontsize=16, fontweight='bold')

axes[0].hist(scores_all, bins=30, color='#185FA5', alpha=0.8, edgecolor='white')
axes[0].axvline(x=0.4, color='#E24B4A', linestyle='--',
                lw=2, label='Acceptable threshold')
axes[0].axvline(x=np.mean(scores_all), color='#1D9E75', linestyle='-',
                lw=2, label=f'Mean={np.mean(scores_all):.2f}')
axes[0].set_xlabel('Quality Score', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Quality Score Distribution', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

rating_counts = {r: ratings.count(r) for r in ['Excellent','Good','Acceptable','Poor']}
axes[1].pie(rating_counts.values(), labels=rating_counts.keys(),
            colors=['#1D9E75','#185FA5','#EF9F27','#E24B4A'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Quality Rating Distribution', fontweight='bold')

sample_img = images[0]
processed  = preprocess_clahe(sample_img)
axes[2].hist(sample_img.flatten(), bins=50, alpha=0.6,
             color='#E24B4A', label='Original', density=True)
axes[2].hist(processed.flatten(), bins=50, alpha=0.6,
             color='#1D9E75', label='After CLAHE', density=True)
axes[2].set_xlabel('Pixel Value', fontsize=12)
axes[2].set_ylabel('Density', fontsize=12)
axes[2].set_title('CLAHE Preprocessing Effect', fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task4_quality_assessment.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n✅ Task 4 完成！已保存：task4_quality_assessment.png')

---
## Week 1 总结

| 步骤 | 数学工具 | 结果 |
|------|---------|------|
| 图像生成 | 高斯分布 + 三角函数 | 500张合成乳腺X光 |
| 特征提取 | 统计/Sobel/LBP/圆度/FFT | (500, 19) 特征矩阵 |
| 标准化 | z = (x-μ)/σ | 统一特征尺度 |
| 随机森林 | 信息熵 + Bootstrap + 投票 | AUC ≈ 1.0 |
| 分析 | Bootstrap CI + 交叉验证 | 全面性能评估 |

**核心洞见：** 100% 准确率反映数据简单性，不代表真实临床能力。  
**下周预告：** CNN 自动学习特征，超越手工设计的上限！

In [ ]:
print('='*60)
print('Week 1 完成！所有输出文件：')
print('  viz_1_sample_images.png')
print('  viz_2_feature_distributions.png')
print('  task1_comprehensive_viz.png')
print('  task2_glcm_features.png')
print('  task3_performance_analysis.png')
print('  task4_quality_assessment.png')
print('='*60)
print(f'最终模型性能：AUC={roc_auc:.3f} | 敏感性={sensitivity:.1%} | 特异性={specificity:.1%}')